# Step 01 — the soft-thresholding power

WGCNA raises every protein–protein correlation to a power, so weak correlations collapse toward
zero and strong ones survive. Everything downstream depends on this one number.

In [ ]:
source("../src/paths.R")
ensure_pkg("WGCNA")               # no conda build for osx-arm64; see src/paths.R
suppressMessages(library(WGCNA))
options(stringsAsFactors = FALSE); enableWGCNAThreads(4)
COHORT <- "A"
X <- read.csv(coh("R_cohort-%s_log2_combat.csv", COHORT), row.names = 1, check.names = FALSE)
sprintf("%d patients x %d proteins", nrow(X), ncol(X))

In [ ]:
sft <- pickSoftThreshold(X, powerVector = c(1:10, seq(12, 20, 2)),
                         networkType = "signed", verbose = 0)
sft$fitIndices[, c("Power", "SFT.R.sq", "slope", "mean.k.")]

## Signed, not unsigned

A **signed** network only joins proteins moving in the *same* direction. Unsigned would place a
protein and its mirror image in one module, which is not a co-expression module in any useful
sense.

This choice is why the automatic power estimate cannot be taken at face value. Signed adjacency
maps a correlation onto [0,1] by `(1+r)/2`, so an **uncorrelated pair starts at 0.5, not 0**, and
needs a much larger exponent before it is pushed away. `pickSoftThreshold` reports the first power
crossing its scale-free criterion, which for a signed network is reliably too permissive — at a low
power almost every pair stays connected and the modules come back as two or three lumps spanning
the whole panel.

The WGCNA authors publish a **floor by sample count** for signed networks — 18 below n=20, 16 to
n=30, 14 to n=40, **12 above** — and it takes precedence over the estimate.

In [ ]:
floor_power <- if (nrow(X) < 20) 18 else if (nrow(X) < 30) 16 else if (nrow(X) < 40) 14 else 12
power <- max(floor_power, ifelse(is.na(sft$powerEstimate), 0, sft$powerEstimate))
sprintf("estimate %s | floor for n=%d is %d | using %d (scale-free R2 = %.2f)",
        sft$powerEstimate, nrow(X), floor_power, power,
        sft$fitIndices$SFT.R.sq[sft$fitIndices$Power == power])

On cohort A the estimate is **10** and the floor is **12**; the fit at 12 is still R² = 0.89, so
nothing is given up by taking the floor.

**This is a defensible choice, not a free one, and it should be stated in any write-up:** the power
was set by the authors' published floor rather than by the automatic estimate, because the automatic
estimate on a signed network produced two modules spanning most of the panel.